# 05 - Normalização para schema canônico

Recebe `configurations_raw.parquet` (17 032 linhas, uma por métrica extraída) e produz
`experiments.parquet` (uma linha por experimento, com todas as métricas como colunas separadas). #TODO

Pipeline neste notebook:
1. Carregamento e merge com ano do paper
2. Parsing numérico (`dataset_size`, `num_classes`, `imbalance_ratio`)
3. Normalização por similaridade: são salvos os raw keys únicos
3.5. LLM auxiliar: mapeia as raw keys para canônico ou none
4. Mapeamento do dataset: aplica-se o mapeamento de raw → canônico para as chaves `dataset_name`, `model_name` e `balancing_strategy` #TODO

In [31]:
import os
import json
import re
from pathlib import Path
from typing import Optional, Literal

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ValidationError

# ── Paths ─────────────────────────────────────────────────────────────────────
NB_DIR        = Path().resolve()
PROJECT_DIR   = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

CKPT_SOURCE  = PROCESSED_DIR / "extraction_checkpoints.jsonl"
INPUT_RAW    = PROCESSED_DIR / "configurations_raw.parquet"
INPUT_PAPERS = PROCESSED_DIR / "papers_with_pdf.parquet"

RAW_DATASETS_OUT = RAW_DIR / "raw_mapped_datasets.jsonl"
RAW_MODELS_OUT = RAW_DIR / "raw_mapped_models.jsonl"
RAW_STRATEGIES_OUT = RAW_DIR / "raw_mapped_strategies.jsonl"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project dir:   {PROJECT_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print("Paths OK")

Project dir:   D:\programas\ufmg\ufmg-2026-1\causal_project
Processed dir: D:\programas\ufmg\ufmg-2026-1\causal_project\data\processed
Paths OK


## 1. Carregamento e merge

In [32]:
df_raw    = pd.read_parquet(INPUT_RAW)
df_papers = pd.read_parquet(INPUT_PAPERS)[["paper_id", "year"]]

df = df_raw.merge(df_papers, on="paper_id", how="left")

print(f"Raw configurations: {len(df):>7,}")
print(f"Unique papers:      {df['paper_id'].nunique():>7,}")
print(f"Year coverage:      {df['year'].notna().mean():.1%}")
print(f"Shape:              {df.shape}")
print()
print("Null rates (key fields):")
for col in ["dataset_name_raw", "model_name_raw", "balancing_strategy_raw",
            "metric_name_raw", "metric_value"]:
    pct = df[col].isna().mean()
    print(f"  {col:<35} {pct:.1%} null")

Raw configurations:  17,032
Unique papers:          512
Year coverage:      100.0%
Shape:              (17032, 20)

Null rates (key fields):
  dataset_name_raw                    0.0% null
  model_name_raw                      0.0% null
  balancing_strategy_raw              0.0% null
  metric_name_raw                     0.0% null
  metric_value                        0.0% null


## 2. Parsing numérico

In [33]:
def parse_dataset_size(s) -> Optional[int]:
    if pd.isna(s) or not s:
        return None
    cleaned = re.sub(r"[,_]", "", str(s))
    m = re.search(r"\d+", cleaned)
    return int(m.group()) if m else None

def parse_num_classes(s) -> Optional[int]:
    if pd.isna(s) or not s:
        return None
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else None

def parse_imbalance_ratio(s) -> Optional[float]:
    """Handles: IR=100, 100:1, 1:100, 100.0 — always returns max/min (≥1)."""
    if pd.isna(s) or not s:
        return None
    raw = str(s)
    # IR=N or IR: N
    m = re.search(r"IR\s*[=:]\s*([\d.]+)", raw, re.I)
    if m:
        return float(m.group(1))
    # N:M ratio
    m = re.search(r"([\d.]+)\s*:\s*([\d.]+)", raw)
    if m:
        a, b = float(m.group(1)), float(m.group(2))
        if min(a, b) > 0:
            return max(a, b) / min(a, b)
    # bare number ≥ 1
    m = re.search(r"[\d.]+", raw)
    if m:
        v = float(m.group())
        return v if v >= 1.0 else None
    return None

df["dataset_size"]            = df["dataset_size_raw"].map(parse_dataset_size)
df["dataset_num_classes"]     = df["dataset_num_classes_raw"].map(parse_num_classes)
df["dataset_imbalance_ratio"] = df["dataset_imbalance_ratio_raw"].map(parse_imbalance_ratio)
df["dataset_is_multilabel"]   = df["task_type_raw"].str.lower().str.contains(
    r"multi.?label", na=False, regex=True
)

print("Numeric parsing results:")
for col in ["dataset_size", "dataset_num_classes", "dataset_imbalance_ratio"]:
    filled = df[col].notna().sum()
    print(f"  {col:<30} {filled:>6,} / {len(df):,}  ({filled/len(df):.1%})")
print(f"  {'dataset_is_multilabel':<30} {df['dataset_is_multilabel'].sum():>6,} rows")

Numeric parsing results:
  dataset_size                    6,010 / 17,032  (35.3%)
  dataset_num_classes             9,201 / 17,032  (54.0%)
  dataset_imbalance_ratio         7,752 / 17,032  (45.5%)
  dataset_is_multilabel             192 rows


## 3. Normalização por regras

In [34]:
unique_datasets_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _d = _cfg.get("dataset_name_raw")
                if _d:
                    unique_datasets_raw.add(_d)
        except Exception:
            pass
        
datasets_json = {}
for dataset in unique_datasets_raw:
    datasets_json[dataset] = []

with open(RAW_DATASETS_OUT, "w", encoding="utf-8") as _f:
    json.dump(datasets_json, _f, ensure_ascii=False, indent=4, sort_keys=True)

print(f"Saved: {RAW_DATASETS_OUT}")
print(f"Total unique dataset names: {len(unique_datasets_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_datasets.jsonl
Total unique dataset names: 962


In [35]:
unique_models_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _m = _cfg.get("model_name_raw")
                if _m:
                    unique_models_raw.add(_m)
        except Exception:
            pass
models_format = {
    "transformer": [],
    "cnn": [],
    "rnn": [],
    "gnn": [],
    "gbm": [],
    "ensemble": [],
    "tree": [],
    "mlp": [],
    "kernel": [],
    "linear": [],
    "other": list(unique_models_raw)
}

with open(RAW_MODELS_OUT, "w", encoding="utf-8") as _f:
    json.dump(models_format, _f, ensure_ascii=False, indent=4)

print(f"Saved: {RAW_MODELS_OUT}")
print(f"Total unique model names: {len(unique_models_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_models.jsonl
Total unique model names: 842


In [36]:
unique_strategies_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _s = _cfg.get("balancing_strategy_raw")
                if _s:
                    unique_strategies_raw.add(_s)
        except Exception:
            pass

strategies_format = {
	"none": [],
	"oversampling": [],
	"undersampling": [],
	"hybrid": [],
	"cost_sensitive": [],
	"data_augmentation": [],
	"ensemble_based": [],
	"threshold_moving": [],
	"generative": [],
	"two_stage": [],
	"other": list(unique_strategies_raw),
}

with open(RAW_STRATEGIES_OUT, "w", encoding="utf-8") as _f:
    json.dump(strategies_format, _f, ensure_ascii=False, indent=4)

print(f"Saved: {RAW_STRATEGIES_OUT}")
print(f"Total unique balancing strategies: {len(unique_strategies_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_strategies.jsonl
Total unique balancing strategies: 1663


## 3.5. Normalização por LLM auxiliar
Foram passados os arquivos raw gerados pelas últimas células para um LLM, com o prompt:  
> "Map all entries to a canonical name (or default keys if available), if possible. If not, keep the original name. Output in JSON".

Os resultados foram salvos em `processed/mapped_datasets.jsonl`, `processed/mapped_models.jsonl` e `processed/mapped_strategies.jsonl`, respectivamente.

## 4. Mapeamento do dataset
Aplica-se o mapeamento de raw → canônico para as chaves `dataset_name`, `model_name` e `balancing_strategy`